# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Thu Oct 22, 2026 - Day 17: Time series and Gaussian processes

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve
from scipy.signal import find_peaks
from astropy.timeseries import LombScargle
from ipywidgets import interactive_output, Dropdown, FloatSlider, IntSlider, ToggleButtons, Label, HBox, VBox, Layout
from IPython.display import display
plt.rcParams.update({'font.size': 13, 'figure.figsize': (9.4, 3.6)})   # 9.4 in = 940 px: fits a 13-inch MacBook mirrored at 1280 px; 7.4 in beside a widget's controls

# matplotlib's math-text parser is a shared singleton and is not thread-safe. ipykernel 7 handles slider messages on
# another thread (ipykernel #1564), so a redraw can overlap the next cell and break any "$...$" label
# ("Expected end of text", ipympl #610). A lock around the parser fixes that on any kernel version (ipympl PR #621).
import threading
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock()
        _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True
        _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")
import ipykernel
if int(ipykernel.__version__.split('.')[0]) >= 7:
    print(f"ipykernel {ipykernel.__version__}: the course pin is 6.31 (INSTALL.txt); the lock above covers the sliders, but plots can silently not appear")

def controls_right(fn, **widgets):
    # the plot on the left, its controls stacked on the right (the Day 12 layout)
    for w in widgets.values():
        w.layout = Layout(width='230px')
        w.style = {'description_width': '75px'}
    display(HBox([interactive_output(fn, widgets), VBox(list(widgets.values()), layout=Layout(justify_content='center', width='240px'))]))

# delta Cephei: AAVSO visual magnitudes, 2019-2020 (lecture/04/data/delcep_lightcurve.csv, the star Day 7 folded on 5.366 d)
_d = pd.read_csv('../04/data/delcep_lightcurve.csv')
t_dc, m_dc = _d['JD'].to_numpy(), _d['Magnitude'].to_numpy()
# 47 UMa: Lick radial velocities, Gregory & Fischer 2010, MNRAS 403, 731 (lecture/07/data/, the Day 13 file)
_g = pd.read_csv('../07/data/gregory2010_47uma_lick_rv.csv', comment='#')
t_rv, rv, rv_e = _g['jd_minus_2440000'].to_numpy(), _g['rv_mps'].to_numpy(), _g['rv_err_mps'].to_numpy()
# NASA GISTEMP v4 global annual mean anomaly, Jan-Dec (lecture/08/data/, the Day 15 file); 2026 is incomplete
_s = pd.read_csv('../08/data/gistemp_v4_GLB.Ts+dSST.csv', skiprows=1)
_s = _s[_s['J-D'] != '***']
yr, anom = _s['Year'].to_numpy(float), _s['J-D'].astype(float).to_numpy()
# a ZTF quasar from Lab 07's live set (ZTF18aawihcn), g band, ALeRCE API copy of 2026-10-21 (data/ztf18aawihcn_g.csv)
_q = pd.read_csv('data/ztf18aawihcn_g.csv', comment='#')
_q = _q[_q['sigma_mag'] < 0.1]                                   # drop a handful of bad-weather points with huge errors
t_q, m_q, e_q = _q['mjd'].to_numpy(), _q['mag_g'].to_numpy(), _q['sigma_mag'].to_numpy()

# Mauna Loa CO2, NOAA GML monthly means (data/co2_mm_mlo_20261022.csv, fetched 2026-10-22; the Rasmussen & Williams Sec. 5.4.3 data set)
_c = pd.read_csv('data/co2_mm_mlo_20261022.csv', comment='#')
t_co2, y_co2 = _c['decimal date'].to_numpy(), _c['average'].to_numpy()

# Lomb-Scargle for delta Cep, and the three highest distinct peaks
f_dc, p_dc = LombScargle(t_dc, m_dc).autopower(minimum_frequency=1/30, maximum_frequency=2, samples_per_peak=20)
P_dc = 1 / f_dc[np.argmax(p_dc)]
FAP_dc = LombScargle(t_dc, m_dc).false_alarm_probability(p_dc.max())
ls_rv = LombScargle(t_rv, rv, rv_e)
f_rv, p_rv = ls_rv.autopower(minimum_frequency=1/5000, maximum_frequency=1/2, samples_per_peak=20)
P_rv = 1 / f_rv[np.argmax(p_rv)]
FAP_rv = ls_rv.false_alarm_probability(p_rv.max())
print(f"delta Cep: {len(t_dc)} points over {(t_dc.max() - t_dc.min()) / 365.25:.2f} yr, LS peak {P_dc:.3f} d, FAP {FAP_dc:.0e}")
print(f"47 UMa: {len(t_rv)} RVs over {(t_rv.max() - t_rv.min()) / 365.25:.1f} yr, LS peak {P_rv:.0f} d, FAP {FAP_rv:.0e}")
print(f"GISTEMP: {len(yr):.0f} annual means, {yr[0]:.0f}-{yr[-1]:.0f}")
print(f"ZTF18aawihcn: {len(t_q)} g-band points over {(t_q.max() - t_q.min()) / 365.25:.1f} yr")

## Quiz 3

* closed everything: pens only, no notes, no phone, no laptop
* 15 minutes, 10 points, 3 questions, all about astronomy problems
* Bayes and MCMC, from Days 12-14: Metropolis, Hamiltonian Monte Carlo, the Bayes factor
* your lowest quiz of the six is dropped
* write on the sheet, put your name on it, and hand it in when I call time

**Time starts when everyone has a sheet.**

## Grades so far (Abha's midterm grades not in yet)

<img src="images/day17_grades_todate.png" alt="Three panels: box plots of Lab 1 to Lab 5 and the review test with most scores near 100 and a few zeros for missing labs; Quiz 1 and Quiz 2 box plots with medians 8.5 and 4.5 out of 10; a histogram of mean lab score with 14 of 19 students above 95" style="display:block;margin:0 auto;max-height:300px">

* labs: 14 of 19 of you average above 95; 9 labs are missing across 5 people
* one free drop, then the late policy - talk to me
* Quiz 2 median 4.5 of 10 against 8.5 on Quiz 1; the lowest of six is dropped


## Today's plan

**When is a peak in a periodogram a period?**

* what Lomb-Scargle (Lab 05) actually does
* aliases, the window function: delta Cephei, 47 UMa
* red noise: a Lab 07 quasar with a "period"
* Gaussian processes: modeling the covariance

Lab 07 posts today; **Clay Miner** and **Pranavan Rajaratnam** defend at 13:40


## Unevenly sampled light curves

* Day 7 folded delta Cephei on 5.366 d; today we make the periodogram
* 1,011 AAVSO points over 14 months, not uniformly sampled - nor can they be, from Earth (rotation, revolution, weather)

**Fourier methods assume uniform sampling - and from Earth, you can't have it.**


In [ ]:
# Day 7's fold, the raw delta Cep light curve, and its sampling (a "skip" cell)
def plot_dc_raw():
    fig, (f, a, b) = plt.subplots(1, 3, figsize=(9.4, 2.9), gridspec_kw={'width_ratios': [1, 1.8, 1]})
    ph7 = (t_dc / 5.366) % 1
    f.plot(ph7, m_dc, 'k.', ms=2, alpha=0.5); f.plot(ph7 + 1, m_dc, 'k.', ms=2, alpha=0.5); f.invert_yaxis()
    f.set_xlabel('phase'); f.set_ylabel('visual magnitude'); f.set_title('Day 7: folded on 5.366 d', fontsize=12)
    a.plot(t_dc - t_dc.min(), m_dc, '.', ms=3)
    a.invert_yaxis(); a.set_xlabel('days since first observation'); a.set_title('the same data, unfolded', fontsize=12)
    gaps = np.diff(np.sort(t_dc))
    b.hist(gaps[gaps > 0], bins=np.logspace(-3, 1.2, 30), color='C1')
    b.set_xscale('log'); b.set_xlabel('gap [d]'); b.set_ylabel('number')
    plt.tight_layout(); plt.show()

In [ ]:
# Day 7's folded curve, the same year of data unfolded, and the spacing of its observations
plot_dc_raw()


* no single sampling frequency: the gaps run from minutes to a week


## Lomb-Scargle: a sinusoid fit at every frequency

Day 13 named it and Lab 05 ran it. Let's take the case where the data are a single sinusoid, $y(t) = a\cos\omega t + b\sin\omega t + c$: linear in $a$ and $b$, so least squares at every trial $\omega$, weighted by $1/\sigma_i^2$.

$$P(\omega) = \frac{\chi^2_0 - \chi^2(\omega)}{\chi^2_0}$$

* $P(\omega)$ is the drop in $\chi^2$ against the constant model, $\chi^2_0$
* Day 5's recipe: a frequency grid, the power at each, take the maximum
* uneven sampling has no Nyquist limit, so the top frequency comes from the physics

**A periodogram is a least-squares fit at every trial frequency - here, about 17,000 of them.**

<div style="font-size:0.75em; color:#666;">VanderPlas 2018, ApJS 236, 16 (arXiv:1703.09824), the "floating mean" form that astropy's LombScargle uses by default.</div>


## EXTRA: there is no Nyquist limit here

* evenly spaced data: nothing above $1/(2\,\Delta t)$, the Nyquist limit from ASTR 310's Fourier unit
* unevenly spaced data: people use $1/\langle \Delta t \rangle$, the median gap, the smallest gap - all wrong
* VanderPlas's example, a star from the LINEAR survey: one point a week on average, and a 2.58-hour period comes out cleanly
* the real limit is how precisely you recorded the times - in practice you stop where the physics stops: apart from white dwarfs, sdB stars and the tightest binaries, which pulse or orbit in minutes, little that is stellar is faster than an hour
* the grid: trust nothing longer than about half the baseline (Lab 07's rule), and space the frequencies $1/(5T)$ or finer so no peak falls between grid points - the setup cell uses 20 samples per peak

**The highest frequency worth trying is a physics decision.**

<div style="font-size:0.75em; color:#666;">VanderPlas 2018, ApJS 236, 16, Sec. 4.1 and 7.1; Eyer &amp; Bartholdi 1999, A&amp;AS 135, 1. Demos after LSSTC-DSFP Session 13 Day 3 (A. A. Miller) and Session 20 Day 1 (S. Blunt).</div>


In [ ]:
# delta Cep periodogram with its labelled peaks, and the light curve folded at any of them (a "skip" cell)
_pk, _ = find_peaks(p_dc)
_pk = _pk[np.argsort(p_dc[_pk])[::-1]][:4]                      # the four highest peaks, by power
dc_peaks = sorted(1 / f_dc[_pk], reverse=True)                  # their periods [d], longest first
_ls_dc = LombScargle(t_dc, m_dc)
def plot_dc_ls(period=None):
    fig, (a, b) = plt.subplots(1, 2, figsize=(9.4, 3.4), gridspec_kw={'width_ratios': [1.4, 1]})
    a.plot(1 / f_dc, p_dc, lw=1)
    a.set_xscale('log'); a.set_xlabel('trial period [d]'); a.set_ylabel('Lomb-Scargle power'); a.set_ylim(top=p_dc.max() * 1.18); a.set_title(f'white-noise false-alarm probability {FAP_dc:.0e}', fontsize=11)
    for n, k in enumerate(_pk):
        a.annotate(f'{1 / f_dc[k]:.3f} d', (1 / f_dc[k], p_dc[k]), textcoords='offset points', xytext=(4, 4 + 12 * (n == 3)), fontsize=11)
    if period is None:
        period = dc_peaks[0]
    a.plot([period], [_ls_dc.power(1 / period)], 'o', color='C3', ms=7)
    ph = ((t_dc - t_dc.min()) / period) % 1
    b.plot(ph, m_dc, '.', ms=3); b.invert_yaxis()
    b.plot(np.linspace(0, 1, 200), _ls_dc.model(t_dc.min() + np.linspace(0, period, 200), 1 / period), 'C3', lw=1.5)
    b.set_xlabel('phase'); b.set_ylabel('visual magnitude'); b.set_title(f'folded on {period:.3f} d, power {_ls_dc.power(1 / period):.2f}', fontsize=12)
    plt.tight_layout(); plt.show()

def dc_peak_buttons():
    # one button per found peak in a row UNDER the plot (a column beside it made the slide taller than the screen); clicking folds exactly there
    j = ToggleButtons(options=[(f'{p:.3f} d', p) for p in dc_peaks], value=dc_peaks[0], style={'button_width': '90px'})
    display(VBox([interactive_output(plot_dc_ls, {'period': j}), HBox([Label('fold at:'), j])]))


## The periodogram of delta Cephei

In [ ]:
# the four highest peaks from peak finding; click one to fold the light curve exactly there
dc_peak_buttons()


<img src="images/day17_static_dc_peaks.png" alt="delta Cep periodogram with four labelled peaks and the light curve folded on the highest, 5.371 d" style="max-height:330px;">

*Static copy of the widget above, for readers on GitHub without running the notebook.*


* any periodic signal in the data will exhibit some power - this includes aliases of the true period. Click each peak: one folds clean
* 0.84 d and 1.23 d are the one-day aliases, $1/P + 1$ and $|1/P - 1|$ (people observe at night!) - Day 5's alias plot

**Here, the peak with the maximum power is the period; the other three are its harmonic (P/2, because the light curve is not a sinusoid) and its aliases.**


## Observe only at night, and every peak gets two neighbours

* we take one sinusoid, 10 h period, 3,000 random times over a year: one peak, 2.4 per day
* night only: two more at 1.4 and 3.4 per day, at 40% of the power
* the window function (the sampling's own periodogram) spikes at 1, 3, 5 per day: the odd harmonics of a half-day square wave


In [ ]:
# A 10-h sinusoid at random times, then night-only, and the window function (a "skip" cell)
# After LSSTC-DSFP Session 20 Day 1, LombScargle1.ipynb Problem 3 (S. Blunt) and VanderPlas 2018 Sec. 3 and 7.2
_rngA = np.random.default_rng(185)
P_sin = 10 / 24.
t_all = np.sort(_rngA.uniform(0, 365, 3000))
y_all = np.sin(2 * np.pi * t_all / P_sin) + 0.3 * _rngA.normal(size=t_all.size)
_night = (t_all % 1) > 0.5                                   # keep the half of every day that is night
t_night, y_night = t_all[_night], y_all[_night]
def plot_night_only():
    fmax = 6.0
    fa, pa = LombScargle(t_all, y_all).autopower(minimum_frequency=1 / 365, maximum_frequency=fmax, samples_per_peak=10)
    fn, pn = LombScargle(t_night, y_night).autopower(minimum_frequency=1 / 365, maximum_frequency=fmax, samples_per_peak=10)
    fw = np.linspace(1 / 365, fmax, 40000)
    pw = LombScargle(t_night, np.ones_like(t_night), fit_mean=False, center_data=False).power(fw)
    fig, ax = plt.subplots(3, 1, figsize=(9.4, 4.4), sharex=True)
    ax[0].plot(fa, pa, lw=0.8); ax[0].set_title(f'{t_all.size} random times, any hour: one peak at 1/P = {1 / P_sin:.2f} per day', fontsize=12)
    ax[1].plot(fn, pn, lw=0.8, color='C3'); ax[1].set_title(f'{t_night.size} of them at night only: the same peak, and aliases at 1/P $\\pm$ 1 (and faintly $\\pm$ 3) per day', fontsize=12)
    for k in [-3, -1, 1, 3]:
        if 1 / P_sin + k > 0: ax[1].axvline(1 / P_sin + k, color='gray', ls=':', lw=1)
    ax[2].plot(fw, pw, lw=0.8, color='C1'); ax[2].set_title('the window function: LombScargle(t, 1, fit_mean=False, center_data=False), spikes at 1, 3, 5 per day', fontsize=12)
    ax[2].set_xlabel('frequency [1/day]')
    for a in ax: a.set_ylabel('power')
    plt.tight_layout(); plt.show()


In [ ]:
# one sinusoid at any hour, then at night only, and the periodogram of the night-only sampling
plot_night_only()


**Every peak leaks into the peaks of the sampling window - night-only observing is a square window, and a square window is very leaky.**


## EXTRA: the phased light curve, free slider

* we can create a phased light curve, $\phi = t/P - \mathrm{int}(t/P)$, and the wrong period scatters it
* try changing the period to see what happens to the fold - and where you are on the periodogram
* drag to 4.0 d, then jump to 5.371 d and to 0.843 d with the buttons


In [ ]:
# Fold on a trial period and mark it on the periodogram (a "skip" cell)
def plot_fold(period):
    fig, (a, b) = plt.subplots(1, 2, figsize=(9.4, 3.4), gridspec_kw={'width_ratios': [1, 1.2]})
    ph = ((t_dc - t_dc.min()) / period) % 1
    a.plot(ph, m_dc, '.', ms=3); a.invert_yaxis()
    _ls = LombScargle(t_dc, m_dc); _tm = t_dc.min() + np.linspace(0, period, 200)
    a.plot(np.linspace(0, 1, 200), _ls.model(_tm, 1 / period), 'C3', lw=1.5)     # the sinusoid the periodogram fit at this period
    a.set_xlabel('phase'); a.set_ylabel('visual magnitude'); a.set_title(f'folded on {period:.3f} d', fontsize=13)
    b.plot(1 / f_dc, p_dc, lw=1)
    pw = LombScargle(t_dc, m_dc).power(1 / period)
    b.plot([period], [pw], 'o', color='C3')
    b.set_xscale('log'); b.set_xlabel('trial period [d]'); b.set_ylabel('power'); b.set_title(f'power {pw:.2f}', fontsize=12)
    plt.tight_layout(); plt.show()

def fold_controls():
    # a free slider, and 'jump to' buttons that set it to a found peak exactly; dragging the slider clears the button
    s = FloatSlider(value=4.0, min=0.5, max=8.0, step=0.001, readout_format='.3f', description='period [d]', layout=Layout(width='330px'), style={'description_width': '75px'})
    j = ToggleButtons(options=[('free', None)] + [(f'{p:.3f} d', p) for p in dc_peaks], value=None, style={'button_width': '80px'})
    j.observe(lambda ch: setattr(s, 'value', ch['new']) if ch['new'] is not None else None, names='value')
    s.observe(lambda ch: setattr(j, 'value', None) if ch['new'] not in dc_peaks else None, names='value')
    display(VBox([interactive_output(plot_fold, {'period': s}), HBox([s, Label('jump to:'), j])]))     # one row under the plot, not a column beside it

In [ ]:
# drag the slider, or click a found peak to jump to it; red curve = the sinusoid that Lomb-Scargle fit
fold_controls()


<img src="images/day17_static_fold.png" alt="delta Cep folded on 5.371 d next to the periodogram with the red dot on the highest peak" style="max-height:330px;">

*Static copy of the widget above, for readers on GitHub without running the notebook.*


* you can get the period by eye from the fold - the periodogram does that at every trial period, and the red dot is its score


## 47 UMa: twenty years of radial velocities

* Day 13: 220 Lick velocities over 21.6 years (Gregory & Fischer 2010), green band at one year
* observing seasons repeat yearly: the window function peaks at one year

<img src="images/day17_day13_47uma_callback.png" alt="Day 13 figure: 47 UMa radial velocities with a flat 0-planet model, the residuals, and their periodogram with a peak near 1000 days and a green band marking one year" style="max-height:120px; display:block; margin:4px auto 0 auto;">


In [ ]:
# 47 UMa RVs, periodogram and window function (a "skip" cell)
def plot_rv():
    fig, (a, b) = plt.subplots(1, 2, figsize=(9.4, 3.0), gridspec_kw={'width_ratios': [1.1, 1]})
    a.errorbar((t_rv + 2440000 - 2451545) / 365.25 + 2000, rv, rv_e, fmt='.', ms=4, lw=0.8)
    a.set_xlabel('year'); a.set_ylabel('radial velocity [m/s]')
    b.plot(1 / f_rv, p_rv, label=f'data (FAP {FAP_rv:.0e})')
    fw = np.linspace(1 / 5000, 1 / 2, 6000)
    pw = LombScargle(t_rv, np.ones_like(rv), fit_mean=False, center_data=False).power(fw)
    b.plot(1 / fw, pw * p_rv.max(), lw=0.8, color='C1', alpha=0.8, label='window (scaled)')
    for _P, _txt, _xy in [(1 / f_rv[np.argmax(p_rv)], 'P', (4, 3)), (273.7, '1/P + 1/yr', (-78, 2)), (563.5, '|1/P - 1/yr|', (6, 2))]:
        _k = np.argmin(np.abs(1 / f_rv - _P)); _k = _k - 15 + np.argmax(p_rv[max(_k - 15, 0):_k + 15])
        b.annotate(f'{_txt}\n{1 / f_rv[_k]:.0f} d', (1 / f_rv[_k], p_rv[_k]), textcoords='offset points', xytext=_xy, fontsize=10)
    b.annotate('window: 1 yr', (370, pw.max() * p_rv.max()), textcoords='offset points', xytext=(-20, 28), fontsize=10, color='C1')
    b.set_ylim(top=p_rv.max() * 1.25)
    b.set_xscale('log'); b.set_xlabel('trial period [d]'); b.set_ylabel('power'); b.legend(fontsize=10, loc='upper left')
    plt.tight_layout(); plt.show()

In [ ]:
# the velocities, and their periodogram over the window: the peak, its two yearly aliases, the 1-yr spike
plot_rv()


* the 1060 d peak is 47 UMa b (Day 13 subtracted Gregory & Fischer's 1079 d orbit)
* 270 d and 560 d are its one-year aliases: $1/270 \approx 1/1060 + 1/365$ and $1/560 \approx 1/365 - 1/1060$, reflected

**Any periodic signal in the data will exhibit some power - here the two extra peaks are the one-year aliases of 47 UMa b.**


## Half the time the highest peak is not the period

* 1,000 sinusoids, periods 0.2-10 d, two years of every third night (60% of the year up, 70% clear): about 100 points each
* Lomb-Scargle gets half within 10%, and the rest fall on two lines


In [ ]:
# 1000 sinusoids on a 3-night survey cadence: Lomb-Scargle peak vs true period (a "skip" cell)
# After LSSTC-DSFP Session 13 Day 3, RealWorldLombScargle.ipynb Problem 3 (A. A. Miller). Cached in data/day17_alias_recovery.npz
import os
_cacheB = 'data/day17_alias_recovery.npz'
if os.path.exists(_cacheB):
    _z = np.load(_cacheB); true_p, ls_p = _z['true_p'], _z['ls_p']
else:
    _rngB = np.random.default_rng(185)
    n_lc = 1000; true_p = np.zeros(n_lc); ls_p = np.zeros(n_lc)
    for lc in range(n_lc):
        t = np.arange(0, 2 * 365, 3) + (_rngB.random(244) / 6 - 1 / 12)                     # every 3rd night, +-4 h jitter
        t = t[np.logical_or(0.3 > t / 365 % 1, t / 365 % 1 > 0.7)]                           # 40% of the year behind the Sun
        t = t[_rngB.random(t.size) <= 0.7]                                                   # 30% weather loss
        true_p[lc] = _rngB.uniform(0.2, 10); amp = _rngB.uniform(1, 5); noise = _rngB.uniform(1, 2)
        yy = amp * np.sin(2 * np.pi * t / true_p[lc]) + _rngB.normal(0, np.sqrt(noise), size=t.size)
        fq, pq = LombScargle(t, yy, np.full(t.size, np.sqrt(noise))).autopower(minimum_frequency=1 / 730, maximum_frequency=12)
        ls_p[lc] = 1 / fq[np.argmax(pq)]
    np.savez_compressed(_cacheB, true_p=true_p, ls_p=ls_p)
frac_ok = np.mean(np.abs(ls_p - true_p) / true_p <= 0.1)
def plot_recovery():
    p_grid = np.linspace(0.1, 10, 1000)
    fig, ax = plt.subplots(figsize=(7.4, 3.4))
    ax.plot(true_p, ls_p, 'o', mfc='None', mec='0.1', alpha=0.5, ms=4)
    for n, lab in [(-1, r'$1/P_{\rm LS} = 1/P - 1/3$'), (1, r'$1/P_{\rm LS} = 1/P + 1/3$')]:
        show = (1 / p_grid + n / 3) > 0
        ax.plot(p_grid[show], 1 / (1 / p_grid[show] + n / 3), lw=1, label=lab)
    ax.set_ylim(0, 12); ax.set_xlabel('true period [d]'); ax.set_ylabel('Lomb-Scargle peak [d]'); ax.legend(fontsize=10, loc='upper left')
    ax.set_title(f'1000 sinusoids on the 3-night cadence: {frac_ok * 100:.0f}% within 10% of the truth', fontsize=12)
    plt.tight_layout(); plt.show()


In [ ]:
# every third night for two years: where the highest peak lands against the true period
plot_recovery()


* the lines are $1/P_{\rm LS} = 1/P \pm 1/3$ per day (the cadence), plus a reflected branch $|1/P - 1/3|$ (the power at $-f$ equals the power at $+f$)
* before you believe a peak: try $2P$ and $3P$, try $f \pm 1$ and $f \pm 2$ per day, fold at each

**Day 5's six nightly points had this problem, and a hundred points on a survey cadence still do - the fix is more data at a different spacing.**


## Significance of the peaks in the periodogram

* false-alarm probability: how often white noise (every point independent of the last) gives a peak this high
* below $10^{-99}$ for delta Cep and 47 UMa: these are variable!

**The significance of a peak depends on your noise model - and so far that model is white noise.**


## A quasar light curve from ZTF

* ZTF18aawihcn (g band, Lab 07's live set): a supermassive black hole's accretion disk, flickering
* the flicker that Samaresh Mondal has to see through to find a tidal disruption event
* the highest peak: is it a period?


In [ ]:
# ZTF quasar light curve and its periodogram (a "skip" cell)
def plot_quasar():
    fig, (a, b) = plt.subplots(1, 2, figsize=(9.4, 2.9), gridspec_kw={'width_ratios': [1.4, 1]})
    a.errorbar((t_q - 51544.5) / 365.25 + 2000, m_q, e_q, fmt='.', ms=3, lw=0.6, color='C2')
    a.invert_yaxis(); a.set_xlabel('year'); a.set_ylabel('ZTF g [mag]'); a.set_title('ZTF18aawihcn', fontsize=12)
    _lsq = LombScargle(t_q, m_q, e_q)
    fq, pq = _lsq.autopower(minimum_frequency=2 / (t_q.max() - t_q.min()), maximum_frequency=1 / 3, samples_per_peak=10)   # longest trial period = half the baseline
    b.plot(1 / fq, pq, lw=0.9, color='C2'); b.set_xscale('log')
    k = np.argmax(pq)
    b.annotate(f'maximum power: {1 / fq[k]:.0f} d', (1 / fq[k], pq[k]), textcoords='offset points', xytext=(-80, 4), fontsize=11)
    b.set_xlabel('trial period [d]'); b.set_ylabel('power'); b.set_ylim(top=pq.max() * 1.2)
    b.set_title(f'white-noise false-alarm probability {_lsq.false_alarm_probability(pq.max()):.0e}', fontsize=12)
    plt.tight_layout(); plt.show()


In [ ]:
# a real quasar: the light curve, and its periodogram with the highest peak labelled
plot_quasar()


* no planet, no pulsation, and the white-noise false-alarm probability says it is certain - but each night depends on the last

**A false-alarm probability is only as good as its noise model - and this noise is not white.**


## Red noise makes periods

* our old friend the random walk, damped: $\phi$ times the last value, plus noise (an AR(1) process)
* memory lasts a characteristic time $\tau$: the autocorrelation function (ACF) is $\phi^{\rm lag}$
* raise $\phi$: the spectrum goes to $1/f^2$ (that is the red), and a long "period" appears
* show $\phi$ = 0, 0.9, 0.95; then seed 5


In [ ]:
# AR(1) noise: series, ACF, and periodogram (a "skip" cell)
def plot_ar1(phi, seed):
    rng = np.random.default_rng(seed)
    n = 600
    e = rng.normal(size=n)
    x = np.zeros(n)
    for i in range(1, n):
        x[i] = phi * x[i - 1] + e[i]
    fig, (a, b, c) = plt.subplots(1, 3, figsize=(7.4, 2.8))
    a.plot(x, lw=0.8); a.set_xlabel('time step'); a.set_ylabel('value')
    xc = x - x.mean()
    acf = np.correlate(xc, xc, 'full')[n - 1:] / (xc @ xc)
    lag = np.arange(60)
    b.plot(lag, acf[:60], 'o', ms=3); b.plot(lag, phi ** lag, 'k-', lw=0.8, label=r'$\phi^{\rm lag}$')
    if 0 < phi < 1: b.axvline(-1 / np.log(phi), color='gray', ls=':', lw=0.8, label=r'$\tau = -1/\ln\phi$')
    b.set_xlabel('lag'); b.set_ylabel('ACF'); b.legend(fontsize=9)
    f, p = LombScargle(np.arange(n), x).autopower(minimum_frequency=1 / n, maximum_frequency=0.5, samples_per_peak=5)
    c.loglog(f, p); c.set_xlabel('frequency [1/step]'); c.set_ylabel('power'); c.set_ylim(bottom=1e-5)
    k = np.argmax(p)
    fig.suptitle(f'phi = {phi:.2f}: max power {p[k]:.2f} at a period of {1 / f[k]:.0f} steps', fontsize=12)
    plt.tight_layout(); plt.show()

In [ ]:
# AR(1) noise with a persistence slider
controls_right(plot_ar1, phi=FloatSlider(value=0.9, min=0.0, max=0.99, step=0.01, description='phi'),
               seed=IntSlider(value=1, min=1, max=20, description='seed'))

<img src="images/day17_static_ar1.png" alt="AR(1) series, ACF and periodogram at phi = 0.95, seed 1" style="max-height:330px;">

*Static copy of the widget above, for readers on GitHub without running the notebook.*


* change the seed: the peak moves - no period, only a correlation time $\tau$

**Stochastic does not mean you cannot characterize this process and make statistical statements about it. The statistic here is $\tau$.**


## Draws from a Gaussian-process prior

* the ACF depends on lag alone, $k(\Delta t)$: the kernel
* show squared exponential, then Matern 1/2; slide $\tau$ 0.2 to 5


In [ ]:
# Sample functions from a GP prior with a chosen kernel (a "skip" cell)
def kern(kind, dt, tau):
    # dt = the lag between two times; tau = the timescale over which the correlation dies (Lab 07's tau)
    r = np.abs(dt) / tau
    if kind == 'squared exponential':
        return np.exp(-0.5 * r ** 2)
    if kind == 'Matern 3/2':
        return (1 + np.sqrt(3) * r) * np.exp(-np.sqrt(3) * r)
    if kind == 'Matern 1/2 (damped random walk)':
        return np.exp(-r)
    return np.exp(-2 * np.sin(np.pi * np.abs(dt) / 4.0) ** 2 / tau ** 2)      # periodic, period fixed at 4

def plot_gp_prior(kind, tau, sigma, seed):
    t = np.linspace(0, 10, 300)
    K = sigma ** 2 * kern(kind, t[:, None] - t[None, :], tau) + 1e-6 * np.eye(t.size)
    L = np.linalg.cholesky(K)
    z = np.random.default_rng(seed).normal(size=(t.size, 5))
    fig, (a, b) = plt.subplots(1, 2, figsize=(7.4, 2.8), gridspec_kw={'width_ratios': [1.8, 1]})
    a.plot(t, L @ z, lw=1.2); a.set_xlabel('t'); a.set_ylabel('f(t)'); a.set_ylim(-3.5, 3.5)
    dt = np.linspace(0, 10, 200)
    b.plot(dt, kern(kind, dt, tau), 'k'); b.set_xlabel(r'lag $\Delta t = |t_i - t_j|$'); b.set_ylabel(r'$k(\Delta t)/k(0)$'); b.set_ylim(-0.05, 1.05)
    plt.tight_layout(); plt.show()


In [ ]:
# five draws from the prior, and the kernel that made them
controls_right(plot_gp_prior,
               kind=Dropdown(options=['squared exponential', 'Matern 3/2', 'Matern 1/2 (damped random walk)',
                                      'periodic (P=4)'], value='squared exponential', description='kernel'),
               tau=FloatSlider(value=1.5, min=0.2, max=5, step=0.1, description='timescale tau'),
               sigma=FloatSlider(value=1.0, min=0.2, max=2, step=0.1, description='amplitude'),
               seed=IntSlider(value=1, min=1, max=20, description='seed'))


<img src="images/day17_static_gp_prior.png" alt="five draws from a Matern 1/2 prior and its kernel" style="max-height:330px;">

*Static copy of the widget above, for readers on GitHub without running the notebook.*


* $\tau$ is the characteristic time, and the kernel's shape sets the smoothness
* Matern 1/2, $k = \sigma^2 \exp(-\Delta t/\tau)$, is the AR(1) again, and Lab 07's damped random walk


## A model for the covariance

$$\ln L = -\tfrac{1}{2}\,\mathbf{r}^T C^{-1}\mathbf{r} - \tfrac{1}{2}\ln|C| - \tfrac{N}{2}\ln 2\pi, \qquad C_{ij} = k(t_i - t_j) + \sigma_{n,i}^2\,\delta_{ij}$$

* Lab 03's likelihood in N-D: $\mathbf{r}$ is the residual vector, and $C$ is no longer diagonal (Day 10 drew it)
* we don't have a parametric model for $y$ anymore, but that's OK - we can write one for the covariance, $k(t_i - t_j)$
* a Gaussian process is completely specified by its mean function and covariance function - $k$ is called the kernel

**We are writing down a model for how the observations are correlated with each other.**


## Conditioning on data

* the prior draws that pass through the data are the posterior, and it is Gaussian too, so its mean and band are matrix algebra (in the notebook, and again in Lab 07)
* training the GP means maximizing $\ln L$ over $\tau$, $\sigma$ and the noise - the $\ln|C|$ term penalizes a short $\tau$ (Day 13's Occam factor)
* inverting $C$ costs $N^3$ (double $N$, eight times slower), so `george` and `celerite` exist


## A GP fit to the global temperature record

* Day 15's GISTEMP file: 146 annual means, 1880-2025; squared-exponential kernel, $\tau$ and noise free
* show $\tau$ = 2 and 80, then back to 24


In [ ]:
# GP regression on the GISTEMP annual means, with the timescale free (a "skip" cell)
_y = anom - anom.mean()
def _kse(a, b, sigma, tau):
    return sigma ** 2 * np.exp(-0.5 * ((a[:, None] - b[None, :]) / tau) ** 2)
def _nll(p, x, y):
    sigma, tau, s = np.exp(p)
    c = cho_factor(_kse(x, x, sigma, tau) + s ** 2 * np.eye(x.size))
    return 0.5 * y @ cho_solve(c, y) + np.log(np.diag(c[0])).sum() + 0.5 * x.size * np.log(2 * np.pi)
_fit = minimize(_nll, np.log([0.5, 20, 0.1]), args=(yr, _y), method='Nelder-Mead')
sig_ml, tau_ml, s_ml = np.exp(_fit.x)
print(f"max-likelihood kernel parameters: amplitude {sig_ml:.2f} K, tau {tau_ml:.0f} yr, noise {s_ml:.2f} K, ln L {-_fit.fun:.1f}")

def plot_gp_fit(tau, noise):
    xs = np.linspace(1880, 2060, 400)
    K = _kse(yr, yr, sig_ml, tau) + noise ** 2 * np.eye(yr.size)
    c = cho_factor(K)
    Ks = _kse(xs, yr, sig_ml, tau)
    mu = Ks @ cho_solve(c, _y)
    var = sig_ml ** 2 - np.einsum('ij,ji->i', Ks, cho_solve(c, Ks.T))
    sd = np.sqrt(np.clip(var, 0, None))
    lnL = -_nll(np.log([sig_ml, tau, noise]), yr, _y)
    fig, ax = plt.subplots(figsize=(7.4, 3.2))
    ax.plot(yr, _y + anom.mean(), 'k.', ms=4, label='GISTEMP annual mean')
    ax.plot(xs, mu + anom.mean(), 'C0', label='GP mean'); ax.fill_between(xs, mu - sd + anom.mean(), mu + sd + anom.mean(), color='C0', alpha=0.3, label=r'$\pm 1\sigma$')
    ax.axvline(yr[-1], color='gray', ls=':')
    ax.set_xlabel('year'); ax.set_ylabel('global anomaly [K]'); ax.legend(fontsize=10, loc='upper left')
    ax.set_title(f'tau = {tau:.0f} yr, noise = {noise:.2f} K, ln L = {lnL:.1f} (best {-_fit.fun:.1f})', fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
# the timescale and the noise level
controls_right(plot_gp_fit,
               tau=FloatSlider(value=float(tau_ml), min=2, max=80, step=1, description='tau [yr]'),
               noise=FloatSlider(value=float(s_ml), min=0.02, max=0.3, step=0.01, description='noise [K]'))


<img src="images/day17_static_gp_fit.png" alt="GP fit to GISTEMP at the maximum-likelihood tau" style="max-height:330px;">

*Static copy of the widget above, for readers on GitHub without running the notebook.*


* note where $\ln L$ is largest!
* past 2025 the mean coasts one more $\tau$, then decays to the prior mean, which we chose to be a constant

**We let the data decide $\tau$, but the prior mean is our choice, and a constant has no trend in it.**


## The same tools, outside astronomy

* your cardiologist runs Lomb-Scargle: heartbeats are unevenly spaced by construction
* Slutzky 1937: a running sum of lottery-ticket digits tracks the 1855-1877 English business cycle, because summed noise looks periodic (our quasar, again)
* GPS stations worldwide show a 351.6 d "year" at 6.5 mm, the satellite constellation's repeat cycle: find the peak, then ask what else has that period
* Rasmussen & Williams' own example, Mauna Loa CO$_2$, 1958 to today (we pulled the live NOAA file this morning)

<div style="font-size:0.75em; color:#666;">Clifford &amp; Tarassenko 2005 (interpolate-then-FFT over-estimates the heart-rate power spectrum); Slutzky 1937; Amiri-Simkooei 2013; Rasmussen &amp; Williams Sec. 5.4.3.</div>


In [ ]:
# GP regression on the Mauna Loa CO2 record: a long trend kernel plus a yearly periodic kernel (a "skip" cell)
# Kernel after Rasmussen & Williams Sec. 5.4.3, their fitted hyperparameters rounded (trend 66 ppm / 67 yr; periodic 2.4 ppm, decay 90 yr, roughness 1.3; noise 0.2 ppm)
def _k_co2(a, b):
    d = a[:, None] - b[None, :]
    trend = 66.0 ** 2 * np.exp(-0.5 * (d / 67.0) ** 2)
    periodic = 2.4 ** 2 * np.exp(-0.5 * (d / 90.0) ** 2) * np.exp(-2 * np.sin(np.pi * d) ** 2 / 1.3 ** 2)
    return trend + periodic
def plot_co2():
    xs = np.linspace(1958, 2040, 1500)
    ym = y_co2.mean()
    c = cho_factor(_k_co2(t_co2, t_co2) + 0.2 ** 2 * np.eye(t_co2.size))
    Ks = _k_co2(xs, t_co2)
    mu = Ks @ cho_solve(c, y_co2 - ym) + ym
    sd = np.sqrt(np.clip(_k_co2(xs, xs).diagonal() - np.einsum('ij,ji->i', Ks, cho_solve(c, Ks.T)), 0, None))
    fig, (a, b) = plt.subplots(1, 2, figsize=(9.4, 2.8), gridspec_kw={'width_ratios': [1.6, 1]})
    a.plot(t_co2, y_co2, 'k.', ms=2, label='NOAA monthly mean'); a.plot(xs, mu, 'C0', lw=1, label='GP mean'); a.fill_between(xs, mu - 2 * sd, mu + 2 * sd, color='C0', alpha=0.3)
    a.axvline(t_co2[-1], color='gray', ls=':'); a.set_xlabel('year'); a.set_ylabel(r'CO$_2$ [ppm]'); a.legend(fontsize=9, loc='upper left')
    a.set_title(f'Mauna Loa, {t_co2.size} months, {y_co2[0]:.0f} to {y_co2[-1]:.0f} ppm', fontsize=11)
    w = (xs > 2024) & (xs < 2032)
    b.plot(xs[w], mu[w], 'C0', lw=1); b.fill_between(xs[w], (mu - 2 * sd)[w], (mu + 2 * sd)[w], color='C0', alpha=0.3)
    b.plot(t_co2[t_co2 > 2024], y_co2[t_co2 > 2024], 'k.', ms=3); b.axvline(t_co2[-1], color='gray', ls=':')
    b.set_xlabel('year'); b.set_title('the yearly kernel keeps the cycle going', fontsize=11)
    plt.tight_layout(); plt.show()


In [ ]:
# the Rasmussen & Williams kernel on this month's NOAA file: trend plus a decaying yearly cycle
plot_co2()


* two kernels added is two processes added: a timescale and an amplitude each
* Rasmussen & Williams let the data set them; here their numbers are typed in

**So how do you choose the kernel? One term per physical process - here a long trend plus a yearly cycle, added.**


## EXTRA: the algebra, for the record

With $K_{ij} = k(t_i - t_j)$ on the data, $K_*$ between the data and the new times $\mathbf{t}_*$, and $K_{**}$ among the new times,

$$\boldsymbol{\mu}_* = K_*^T (K + \sigma_n^2 I)^{-1}\mathbf{y}, \qquad \Sigma_* = K_{**} - K_*^T (K + \sigma_n^2 I)^{-1} K_*$$

<div style="font-size:0.75em; color:#666;">Rasmussen &amp; Williams 2006, <i>Gaussian Processes for Machine Learning</i>, Eqs. 2.23-2.24 (the prediction) and 2.30 (the likelihood); free at gaussianprocess.org/gpml.</div>


## Wrapping up: when is a peak a period?

* Lomb-Scargle fits a sinusoid at each frequency - compute the window function first
* a peak's significance depends on your noise model; white noise is rarely defensible
* we are writing down a model for how the observations are correlated with each other
* timescale and amplitude are parameters: maximize $\ln L$ or sample (Lab 05); Lab 07 does it on a quasar

Reading: Ivezic et al., *Statistics, Data Mining, and Machine Learning in Astronomy*, Ch. 10 and Sec. 8.10. Rasmussen and Williams, Ch. 2, free at gaussianprocess.org/gpml


## Tuesday's colloquium speaker, 3:45 pm: Jiayi Sun (Kentucky)

<img src="images/speaker_jiayi_sun.jpg" alt="Portrait of Jiayi Sun" style="float:right; max-height:200px; margin:0 0 8px 16px;">

* **"Molecular Cloud Scale 'Micro-physics' as Drivers of the Baryon Cycle and Galaxy Evolution"**
* the talk builds populations of molecular clouds across nearby galaxies, a population model, which is where Tuesday's hierarchical Bayes goes
* 3:45 pm, Kaler Classroom (134 Astronomy, this room); refreshments 3:30 pm, Astronomy 222

<div style="font-size:0.75em; color:#666;">Listing: astro.illinois.edu/news-events/astrophysics-colloquium and calendars.illinois.edu (title checked 2026-10-21). Photo: University of Kentucky Physics &amp; Astronomy faculty page.</div>


## Announcements

* **Clay Miner** and **Pranavan Rajaratnam** defend now; everyone stays
* no **capstone topic** yet? Find me today
* Lab 07 posted, due Wed Oct 28 by Noon; dates in `ASSIGNMENTS.md`
* nothing collected today but Quiz 3
* Tuesday, Oct 27: probabilistic graphical models, hierarchical Bayes; Jiayi Sun's colloquium at 3:45
